In [2]:
!pip install -q datasets transformers torch

In [10]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

# ---------------------------------------------------------
# 1. Load IIT Bombay Dataset from Hugging Face
# ---------------------------------------------------------
print("Loading IITB English-Hindi dataset...")
dataset = load_dataset("cfilt/iitb-english-hindi")

# Subsetting 50k samples for quick execution on Kaggle
# Remove .select(range(50000)) if training on all 1.6M+ pairs
train_ds = dataset["train"].select(range(50000))
val_ds = dataset["validation"]

print(f"Loaded {len(train_ds)} train pairs | {len(val_ds)} val pairs")

# ---------------------------------------------------------
# 2. Tokenizer Setup (English -> Hindi)
# ---------------------------------------------------------
# Helsinki-NLP tokenizer handles Devanagari script tokenization smoothly
model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

SOURCE_LANG = "en"
TARGET_LANG = "hi"
MAX_LENGTH = 128

# ---------------------------------------------------------
# 3. Preprocessing (Text -> Integer IDs)
# ---------------------------------------------------------
def preprocess_function(examples):
    # Extract source (English) and target (Hindi) strings
    inputs = [ex[SOURCE_LANG] for ex in examples["translation"]]
    targets = [ex[TARGET_LANG] for ex in examples["translation"]]
    
    # Tokenize English (Encoder Inputs)
    model_inputs = tokenizer(
        inputs, 
        max_length=MAX_LENGTH, 
        truncation=True, 
        padding="max_length"
    )
    
    # Tokenize Hindi (Decoder Targets)
    labels = tokenizer(
        text_target=targets, 
        max_length=MAX_LENGTH, 
        truncation=True, 
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing sentences...")
tokenized_train = train_ds.map(
    preprocess_function, 
    batched=True, 
    remove_columns=train_ds.column_names
)
tokenized_val = val_ds.map(
    preprocess_function, 
    batched=True, 
    remove_columns=val_ds.column_names
)

# Convert mapped lists to PyTorch Tensors
tokenized_train.set_format(type="torch")
tokenized_val.set_format(type="torch")

# ---------------------------------------------------------
# 4. PyTorch DataLoaders (Mini-Batch Gradient Descent)
# ---------------------------------------------------------
BATCH_SIZE = 64

train_loader = DataLoader(tokenized_train, batch_size=BATCH_SIZE, shuffle=True,num_workers=2,pin_memory=True)
val_loader = DataLoader(tokenized_val, batch_size=BATCH_SIZE, shuffle=False,num_workers=2,pin_memory=True)

# ---------------------------------------------------------
# 5. Verify Tensor Shapes
# ---------------------------------------------------------
batch = next(iter(train_loader))

print("\n--- Tensor Shapes for Non-Attention Seq2Seq ---")
print("Encoder Input IDs (`input_ids`):", batch["input_ids"].shape) # [Batch_Size, Max_Len]
print("Decoder Target IDs (`labels`):   ", batch["labels"].shape)    # [Batch_Size, Max_Len]

Loading IITB English-Hindi dataset...
Loaded 50000 train pairs | 520 val pairs
Tokenizing sentences...

--- Tensor Shapes for Non-Attention Seq2Seq ---
Encoder Input IDs (`input_ids`): torch.Size([64, 128])
Decoder Target IDs (`labels`):    torch.Size([64, 128])


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD_IDX = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
BOS_IDX = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else PAD_IDX
EOS_IDX = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_IDX


class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_IDX
        )
        self.LSTM = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
        )

    def forward(self, text):
        embeded = self.embedding(text)
        out, (hidden, cell) = self.LSTM(embeded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_IDX
        )
        self.LSTM = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, text, hidden, cell):
        embeded = self.embedding(text)
        out, (hidden, cell) = self.LSTM(embeded, (hidden, cell))
        probabilities = self.fc(out)
        return probabilities, hidden, cell


vocab_size = tokenizer.vocab_size
embeded_dim = 256
hidden_size = 512
patience = 2
patience_cout = 0
best_val_loss = float("inf")
learning_rate = 0.001
num_epoch = 3

wandb.init(
    project="seq2seq-en-hi",
    name="lstm-baseline",
    config={
        "learning_rate": learning_rate,
        "epochs": num_epoch,
        "batch_size": 64,
        "embedding_dim": embeded_dim,
        "hidden_size": hidden_size,
        "architecture": "Seq2Seq-LSTM",
        "dataset": "IITB-En-Hi-50k",
    },
)

encoder = Encoder(vocab_size, embeded_dim, hidden_size).to(device)
decoder = Decoder(vocab_size, embeded_dim, hidden_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

all_params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(params=all_params, lr=learning_rate)

for epoch in range(num_epoch):
    epoch_loss = 0.0
    encoder.train()
    decoder.train()

    for batch in train_loader:
        input = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        labels[labels == -100] = PAD_IDX
        batch_size = labels.size(0)

        bos_tokens = torch.full((batch_size, 1), BOS_IDX, device=device)
        dec_input = torch.cat([bos_tokens, labels[:, :-1]], dim=1)
        dec_target = labels

        hidden, cell = encoder(input)
        out, _, _ = decoder(dec_input, hidden, cell)

        optimizer.zero_grad()
        loss = criterion(
            out.reshape(-1, tokenizer.vocab_size), dec_target.reshape(-1)
        )

        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch: {epoch} Loss: {train_epoch_loss:.4f}")
    wandb.log(
        {
            "epoch": epoch,
            "train_loss": train_epoch_loss,
        }
    )

# Save trained models
torch.save(encoder.state_dict(), "encoder_final.pt")
torch.save(decoder.state_dict(), "decoder_final.pt")

# ==========================================
# BLEU EVALUATION CODE
# ==========================================
print("\nEvaluating BLEU Score...")

encoder.eval()
decoder.eval()

references = []
hypotheses = []
smooth = SmoothingFunction().method1
max_gen_len = 50

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        labels[labels == -100] = PAD_IDX

        b_size = input_ids.size(0)
        hidden, cell = encoder(input_ids)

        dec_input = torch.full((b_size, 1), BOS_IDX, device=device)
        generated_tokens = []

        for _ in range(max_gen_len):
            out, hidden, cell = decoder(dec_input, hidden, cell)
            next_token = out.argmax(dim=-1)  # Shape: (batch_size, 1)
            generated_tokens.append(next_token)
            dec_input = next_token

        pred_seqs = torch.cat(generated_tokens, dim=1).cpu().tolist()
        target_seqs = labels.cpu().tolist()

        for pred, target in zip(pred_seqs, target_seqs):
            pred_clean = []
            for tok in pred:
                if tok in (PAD_IDX, EOS_IDX):
                    break
                pred_clean.append(tok)

            target_clean = [
                tok for tok in target if tok not in (PAD_IDX, BOS_IDX, EOS_IDX)
            ]

            pred_words = tokenizer.decode(pred_clean).strip().split()
            target_words = tokenizer.decode(target_clean).strip().split()

            hypotheses.append(pred_words)
            references.append([target_words])

final_bleu = corpus_bleu(references, hypotheses, smoothing_function=smooth) * 100.0
print(f"Final BLEU Score: {final_bleu:.2f}")
wandb.log({"final_bleu": final_bleu})

wandb.finish()

Epoch: 0 Loss: 4.8494
Epoch: 1 Loss: 2.7815
Epoch: 2 Loss: 1.7707

Evaluating BLEU Score...
Final BLEU Score: 0.01


epoch,▁▅█
final_bleu,▁
train_loss,█▃▁
epoch,2
final_bleu,0.00897
train_loss,1.77071


In [26]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer

# =====================================================
# DEVICE
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# TOKENIZER
# =====================================================

model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

PAD_IDX = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
BOS_IDX = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else PAD_IDX
EOS_IDX = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_IDX

MAX_LENGTH = 128

# =====================================================
# ENCODER
# =====================================================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.LSTM = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
        )

    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, cell) = self.LSTM(embedded)
        return hidden, cell

# =====================================================
# DECODER
# =====================================================

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        self.LSTM = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, text, hidden, cell):

        embedded = self.embedding(text)

        out, (hidden, cell) = self.LSTM(
            embedded,
            (hidden, cell)
        )

        out = self.fc(out)

        return out, hidden, cell

# =====================================================
# LOAD MODEL
# =====================================================

embedding_dim = 256
hidden_size = 512
vocab_size = tokenizer.vocab_size

encoder = Encoder(
    vocab_size,
    embedding_dim,
    hidden_size
).to(device)

decoder = Decoder(
    vocab_size,
    embedding_dim,
    hidden_size
).to(device)

encoder.load_state_dict(
    torch.load("encoder_final.pt", map_location=device)
)

decoder.load_state_dict(
    torch.load("decoder_final.pt", map_location=device)
)

encoder.eval()
decoder.eval()

print("Models Loaded Successfully!")

# =====================================================
# TRANSLATION FUNCTION
# =====================================================

def translate(sentence, max_len=50):

    encoded = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    input_ids = encoded["input_ids"].to(device)

    with torch.no_grad():

        hidden, cell = encoder(input_ids)

        decoder_input = torch.tensor([[BOS_IDX]], device=device)

        generated = []

        for _ in range(max_len):

            output, hidden, cell = decoder(
                decoder_input,
                hidden,
                cell
            )

            next_token = output.argmax(dim=-1)

            token = next_token.item()

            if token == EOS_IDX:
                break

            generated.append(token)

            decoder_input = next_token

    translation = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return translation

# =====================================================
# INTERACTIVE LOOP
# =====================================================
# =====================================================
# TEST SENTENCES
# =====================================================

sentence = "How are you?"

encoded = tokenizer(
    sentence,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(device)

with torch.no_grad():

    hidden, cell = encoder(input_ids)

    decoder_input = torch.tensor([[PAD_IDX]], device=device)

    output, _, _ = decoder(decoder_input, hidden, cell)

    probs = torch.softmax(output[0, 0], dim=0)

    values, indices = torch.topk(probs, 10)

for p, idx in zip(values, indices):
    print(idx.item(), tokenizer.decode([idx.item()]), p.item())

Models Loaded Successfully!
1228 ~ 0.11443236470222473
599 चयनित 0.11172201484441757
60 यह 0.0546836294233799
1303 आपने 0.04439837485551834
681 कृपया 0.03663770854473114
89 इस 0.03226163238286972
29 नहीं 0.031173383817076683
261 यदि 0.030447334051132202
44  0.020595289766788483
1939 अतिरिक्त 0.020295143127441406


In [23]:
print(tokenizer.decode([BOS_IDX]))
print(tokenizer.decode([EOS_IDX]))
print(PAD_IDX, BOS_IDX, EOS_IDX)
print(tokenizer.special_tokens_map)

<pad>
</s>
61949 61949 0
{'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}


In [25]:
print(train_ds[0])

sample = preprocess_function(train_ds[:1])

print(tokenizer.decode(sample["input_ids"][0], skip_special_tokens=False))
print(tokenizer.decode(sample["labels"][0], skip_special_tokens=False))

batch = next(iter(train_loader))
print(batch["labels"][0][:20])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}
Give your application an accessibility workout</s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें</s> <pad> <pad> <pad> <pad> <pad> <pad> <pad